# Module 3 Instructor Colab Notebook: Eigenvalues, Eigenvectors, and Eigendecomposition

## A Mathematical Bridge to Artificial Intelligence

This module studies special directions that a matrix scales without turning. Those directions make repeated matrix transformations easier to understand and provide the mathematical foundation for principal component analysis and part of the stability story for recurrent neural networks.

**Recommended pacing:** two or three class meetings. The PCA and recurrent-network sections may be taught as separate application days.


> **How to use this notebook in class**
>
> Require students to verify every proposed eigenpair directly before solving characteristic equations. Use the visual cells to distinguish “a vector is transformed” from “an eigenvector keeps its direction.” In the applications, emphasize both the usefulness and the limitations of eigenvalue reasoning.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Eigenvalues and eigenvectors | Special directions of a linear transformation |
| 2 | Characteristic equation | Systematic computation of eigenpairs |
| 3 | Eigendecomposition | Efficient repeated transformations |
| 4 | Symmetric matrices and PCA | Finding directions of greatest variation |
| 5 | Repeated dynamics and spectral radius | Vanishing and exploding behavior in RNNs |
| Integrated lab | Eigenmode decomposition | Predicting a recurrent state exactly |


## Module learning objectives

By the end of Module 3, students should be able to:

1. explain geometrically what makes an eigenvector special;
2. define eigenvalue and eigenvector and explain why an eigenvector must be nonzero;
3. verify an eigenpair using direct multiplication;
4. derive and solve the characteristic equation;
5. find an eigenvector basis associated with each eigenvalue;
6. distinguish algebraic multiplicity from geometric multiplicity;
7. construct an eigendecomposition when enough independent eigenvectors exist;
8. use diagonalization to compute powers of a matrix;
9. explain the centering, covariance, eigenvector, and projection steps of PCA;
10. use eigenvalue magnitude and spectral radius to analyze repeated linear dynamics while recognizing important limitations.


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable display and instructor-reveal helpers.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def annotated_heatmap(matrix, title, row_labels=None, column_labels=None, cmap="Blues"):
    matrix = np.asarray(matrix, dtype=float)
    fig, ax = plt.subplots(figsize=(max(5, matrix.shape[1] * 1.2),
                                    max(3.5, matrix.shape[0] * 0.8)))
    image = ax.imshow(matrix, cmap=cmap, aspect="auto")
    if row_labels is not None:
        ax.set_yticks(range(matrix.shape[0]), row_labels)
    if column_labels is not None:
        ax.set_xticks(range(matrix.shape[1]), column_labels)
    ax.set_title(title)
    threshold = (matrix.max() + matrix.min()) / 2 if matrix.size else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            color = "white" if matrix[i, j] > threshold else "black"
            ax.text(j, i, f"{matrix[i,j]:g}", ha="center", va="center", color=color)
    fig.colorbar(image, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


> **Notation convention**
>
> The Greek letter lambda denotes an eigenvalue. Bold lowercase letters denote vectors. The zero vector is never called an eigenvector. Numerical software may return eigenvectors with different signs or scales; those vectors represent the same eigendirection.

---

# Section 1: Eigenvalues and Eigenvectors

## Learning objectives

Students should be able to identify direction-preserving vectors, verify eigenpairs, interpret positive, negative, and zero eigenvalues, and prove how an eigenmode behaves under repeated multiplication.


## 1. Background: how a matrix transforms vectors

A square matrix defines a linear transformation from a space back to itself:

$$
A\in\mathbb{R}^{n\times n},
\qquad
\mathbf{x}\in\mathbb{R}^{n},
\qquad
A\mathbf{x}\in\mathbb{R}^{n}.
$$

Most input vectors change both length and direction. However, some nonzero directions are special: the transformation only stretches, shrinks, reverses, or collapses them. These are eigendirections.


## 2. Definition of eigenvalue and eigenvector

Let

$$
A\in\mathbb{R}^{n\times n}.
$$

A nonzero vector

$$
\mathbf{v}\neq\mathbf{0}
$$

is an **eigenvector** of `A` if there is a scalar `lambda` such that

$$
A\mathbf{v}=\lambda\mathbf{v}.
$$

The scalar `lambda` is the corresponding **eigenvalue**.

The equation says that the output is a scalar multiple of the input. Therefore, the output remains on the same line through the origin.

| Eigenvalue behavior | Geometric effect on the eigenvector |
|---|---|
| Greater than `1` | stretch without reversing |
| Between `0` and `1` | shrink without reversing |
| Equal to `1` | preserve the vector |
| Equal to `0` | send the eigenvector to zero |
| Negative | reverse direction and scale by the absolute value |


> **Why the eigenvector must be nonzero**
>
> The zero vector satisfies
>
> $$
> A\mathbf{0}=\lambda\mathbf{0}
> $$
>
> for every scalar `lambda`. If the zero vector were allowed, it would provide no information about a special direction and every number would appear to be an eigenvalue. Requiring a nonzero vector makes the definition meaningful.


## 3. First detailed example: a diagonal matrix

Let

$$
D=
\begin{bmatrix}
3&0\\
0&-2
\end{bmatrix}.
$$

For the first coordinate direction,

$$
\mathbf{e}_1=
\begin{bmatrix}
1\\0
\end{bmatrix},
$$

we obtain

$$
D\mathbf{e}_1
=
\begin{bmatrix}
3&0\\0&-2
\end{bmatrix}
\begin{bmatrix}
1\\0
\end{bmatrix}
=
\begin{bmatrix}
3\\0
\end{bmatrix}
=3\mathbf{e}_1.
$$

Thus `e1` is an eigenvector with eigenvalue `3`.

For

$$
\mathbf{e}_2=
\begin{bmatrix}
0\\1
\end{bmatrix},
$$

$$
D\mathbf{e}_2
=
\begin{bmatrix}
0\\-2
\end{bmatrix}
=-2\mathbf{e}_2.
$$

Thus `e2` is an eigenvector with eigenvalue `-2`. The negative eigenvalue reverses the second coordinate direction.


In [ ]:
D_eigen = np.array([[3.0, 0.0], [0.0, -2.0]])
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])
print("D e1:", D_eigen @ e1, "= 3 e1")
print("D e2:", D_eigen @ e2, "= -2 e2")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, vector, transformed, title in [
    (axes[0], e1, D_eigen@e1, "Positive eigenvalue: stretch"),
    (axes[1], e2, D_eigen@e2, "Negative eigenvalue: reverse and stretch"),
]:
    ax.quiver(0, 0, vector[0], vector[1], angles="xy", scale_units="xy", scale=1,
              width=0.015, color="#4c78a8", label="v")
    ax.quiver(0, 0, transformed[0], transformed[1], angles="xy", scale_units="xy", scale=1,
              width=0.012, color="#e45756", label="Av")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()


## 4. A non-diagonal example with two eigendirections

Let

$$
A=
\begin{bmatrix}
2&1\\
1&2
\end{bmatrix}.
$$

Test

$$
\mathbf{v}_1=
\begin{bmatrix}
1\\1
\end{bmatrix}.
$$

$$
A\mathbf{v}_1
=
\begin{bmatrix}
2(1)+1(1)\\
1(1)+2(1)
\end{bmatrix}
=
\begin{bmatrix}
3\\3
\end{bmatrix}
=3
\begin{bmatrix}
1\\1
\end{bmatrix}.
$$

Thus `v1` has eigenvalue `3`.

Now test

$$
\mathbf{v}_2=
\begin{bmatrix}
1\\-1
\end{bmatrix}.
$$

$$
A\mathbf{v}_2
=
\begin{bmatrix}
2(1)+1(-1)\\
1(1)+2(-1)
\end{bmatrix}
=
\begin{bmatrix}
1\\-1
\end{bmatrix}
=1\mathbf{v}_2.
$$

Thus `v2` has eigenvalue `1`.


In [ ]:
A_direction = np.array([[2.0, 1.0], [1.0, 2.0]])
v1_direction = np.array([1.0, 1.0])
v2_direction = np.array([1.0, -1.0])
ordinary_vector = np.array([1.0, 0.25])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, vector, title in zip(
    axes,
    [v1_direction, v2_direction, ordinary_vector],
    ["Eigenvector: eigenvalue 3", "Eigenvector: eigenvalue 1", "Ordinary vector: direction changes"],
):
    transformed = A_direction @ vector
    ax.quiver(0, 0, vector[0], vector[1], angles="xy", scale_units="xy", scale=1,
              width=0.015, color="#4c78a8", label="input")
    ax.quiver(0, 0, transformed[0], transformed[1], angles="xy", scale_units="xy", scale=1,
              width=0.012, color="#e45756", label="output")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()


## 5. Eigenvectors come in families

If

$$
A\mathbf{v}=\lambda\mathbf{v},
$$

then for any nonzero scalar `c`,

$$
A(c\mathbf{v})
=cA\mathbf{v}
=c\lambda\mathbf{v}
=\lambda(c\mathbf{v}).
$$

Therefore, every nonzero scalar multiple of an eigenvector is another eigenvector with the same eigenvalue. An **eigendirection** is the line through all those multiples.

This is why software may return

$$
\begin{bmatrix}0.7071\\0.7071\end{bmatrix}
$$

instead of

$$
\begin{bmatrix}1\\1\end{bmatrix}.
$$

They point in the same direction and represent the same eigenvector family.


## 6. Repeated action on an eigenvector

If

$$
A\mathbf{v}=\lambda\mathbf{v},
$$

then

$$
A^k\mathbf{v}=\lambda^k\mathbf{v}
$$

for every positive integer `k`.

### Proof by induction

For `k=1`, the statement is the eigenvector definition.

Assume

$$
A^k\mathbf{v}=\lambda^k\mathbf{v}.
$$

Then

$$
A^{k+1}\mathbf{v}
=A(A^k\mathbf{v}).
$$

Use the induction assumption:

$$
=A(\lambda^k\mathbf{v}).
$$

Use linearity:

$$
=\lambda^kA\mathbf{v}.
$$

Use the eigenvector equation:

$$
=\lambda^k(\lambda\mathbf{v})
=\lambda^{k+1}\mathbf{v}.
$$

Thus the result holds for all positive integers.


> **Common mistakes in Section 1**
>
> - Calling the zero vector an eigenvector.
> - Checking only that the output has the same dimension; it must be a scalar multiple of the input.
> - Treating every vector as an eigenvector.
> - Assuming eigenvectors must have length one.
> - Believing a negative eigenvalue means the vector is invalid.
> - Confusing an eigenvalue with an entry of the matrix.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Linear transformation | Map represented by multiplication by a matrix |
| Eigenvector | Nonzero direction scaled but not turned by a matrix |
| Eigenvalue | Scaling factor associated with an eigenvector |
| Eigenpair | An eigenvalue together with a corresponding eigenvector |
| Eigenspace | Set of all eigenvectors for one eigenvalue together with zero |
| Eigendirection | Line through scalar multiples of an eigenvector |


## Section 1 worksheet

For

$$
A=
\begin{bmatrix}
4&0\\
0&0.5
\end{bmatrix},
$$

1. verify that the coordinate directions are eigenvectors;
2. state their eigenvalues;
3. compute the result of applying `A` five times to each coordinate direction;
4. explain which direction grows faster and why;
5. determine whether

$$
\begin{bmatrix}1\\1\end{bmatrix}
$$

is an eigenvector.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
### 1–2. Coordinate eigenvectors

$$
A\begin{bmatrix}1\\0\end{bmatrix}
=\begin{bmatrix}4\\0\end{bmatrix}
=4\begin{bmatrix}1\\0\end{bmatrix},
$$

so the first eigenvalue is `4`.

$$
A\begin{bmatrix}0\\1\end{bmatrix}
=\begin{bmatrix}0\\0.5\end{bmatrix}
=0.5\begin{bmatrix}0\\1\end{bmatrix},
$$

so the second eigenvalue is `0.5`.

### 3. Five applications

$$
A^5\mathbf{e}_1=4^5\mathbf{e}_1=1024\mathbf{e}_1,
$$

$$
A^5\mathbf{e}_2=(0.5)^5\mathbf{e}_2=\frac{1}{32}\mathbf{e}_2.
$$

### 4. Growth

The first direction grows because its eigenvalue magnitude exceeds one. The second shrinks because its eigenvalue magnitude is below one.

### 5. Non-eigenvector

$$
A\begin{bmatrix}1\\1\end{bmatrix}
=\begin{bmatrix}4\\0.5\end{bmatrix}.
$$

The two coordinates were scaled by different factors, so the output is not a scalar multiple of the input. The vector is not an eigenvector.
""")


---

# Section 2: Characteristic Equation and Computing Eigenpairs

## Learning objectives

Students should be able to derive the characteristic equation, compute eigenvalues using a determinant, solve for corresponding eigenvectors, and interpret algebraic and geometric multiplicities.


## 1. From the eigenvector equation to a homogeneous system

Start with

$$
A\mathbf{v}=\lambda\mathbf{v}.
$$

Insert the identity matrix:

$$
A\mathbf{v}=\lambda I\mathbf{v}.
$$

Move everything to one side:

$$
A\mathbf{v}-\lambda I\mathbf{v}=\mathbf{0}.
$$

Factor out the vector:

$$
(A-\lambda I)\mathbf{v}=\mathbf{0}.
$$

We need a nonzero solution. A square homogeneous system has a nonzero solution exactly when its coefficient matrix is singular, meaning its determinant is zero. Therefore,

$$
\det(A-\lambda I)=0.
$$

This is the **characteristic equation**. The polynomial

$$
p_A(\lambda)=\det(A-\lambda I)
$$

is the **characteristic polynomial**.


## 2. Determinant review for a two-by-two matrix

For

$$
M=
\begin{bmatrix}
a&b\\c&d
\end{bmatrix},
$$

the determinant is

$$
\det(M)=ad-bc.
$$

The determinant is zero precisely when the columns are dependent. The matrix is then not invertible: distinct inputs can map to the same output, and the null space contains a nonzero vector. That is why a zero determinant permits a nonzero solution of

$$
M\mathbf{v}=\mathbf{0}.
$$


## 3. Detailed characteristic-equation example

Let

$$
A=
\begin{bmatrix}
4&1\\
2&3
\end{bmatrix}.
$$

First form

$$
A-\lambda I
=
\begin{bmatrix}
4&1\\2&3
\end{bmatrix}
-
\begin{bmatrix}
\lambda&0\\0&\lambda
\end{bmatrix}
=
\begin{bmatrix}
4-\lambda&1\\
2&3-\lambda
\end{bmatrix}.
$$

Compute the determinant:

$$
\det(A-\lambda I)
=(4-\lambda)(3-\lambda)-2(1).
$$

Expand:

$$
(4-\lambda)(3-\lambda)
=12-4\lambda-3\lambda+\lambda^2.
$$

Therefore,

$$
p_A(\lambda)
=\lambda^2-7\lambda+12-2.
$$

$$
=\lambda^2-7\lambda+10.
$$

Factor:

$$
\lambda^2-7\lambda+10
=(\lambda-5)(\lambda-2).
$$

The eigenvalues are

$$
\lambda_1=5,
\qquad
\lambda_2=2.
$$


## 4. Eigenvector for eigenvalue five

Solve

$$
(A-5I)\mathbf{v}=\mathbf{0}.
$$

$$
A-5I
=
\begin{bmatrix}
-1&1\\
2&-2
\end{bmatrix}.
$$

Let

$$
\mathbf{v}=
\begin{bmatrix}x\\y\end{bmatrix}.
$$

The first row gives

$$
-x+y=0,
$$

so

$$
y=x.
$$

Choose the free value

$$
x=1.
$$

Then

$$
\mathbf{v}_1=
\begin{bmatrix}1\\1\end{bmatrix}.
$$

Verification:

$$
A\mathbf{v}_1
=
\begin{bmatrix}5\\5\end{bmatrix}
=5\mathbf{v}_1.
$$


## 5. Eigenvector for eigenvalue two

Solve

$$
(A-2I)\mathbf{v}=\mathbf{0}.
$$

$$
A-2I
=
\begin{bmatrix}
2&1\\
2&1
\end{bmatrix}.
$$

The equation is

$$
2x+y=0.
$$

Therefore,

$$
y=-2x.
$$

Choose

$$
x=1.
$$

Then

$$
\mathbf{v}_2=
\begin{bmatrix}1\\-2\end{bmatrix}.
$$

Verification:

$$
A\mathbf{v}_2
=
\begin{bmatrix}2\\-4\end{bmatrix}
=2\mathbf{v}_2.
$$


In [ ]:
A_char = np.array([[4.0, 1.0], [2.0, 3.0]])
eigenvalues_char, eigenvectors_char = np.linalg.eig(A_char)

print("NumPy eigenvalues:", eigenvalues_char)
print("NumPy unit eigenvectors stored as columns:\n", eigenvectors_char)

v_lambda5 = np.array([1.0, 1.0])
v_lambda2 = np.array([1.0, -2.0])
print("A v1 - 5v1:", A_char@v_lambda5 - 5*v_lambda5)
print("A v2 - 2v2:", A_char@v_lambda2 - 2*v_lambda2)


## 6. Trace and determinant checks

For a two-by-two matrix, the characteristic polynomial can be written as

$$
\lambda^2-\operatorname{tr}(A)\lambda+\det(A).
$$

The **trace** is the sum of diagonal entries:

$$
\operatorname{tr}(A)=4+3=7.
$$

The determinant is

$$
\det(A)=4(3)-1(2)=10.
$$

The eigenvalues satisfy

$$
\lambda_1+\lambda_2=\operatorname{tr}(A),
$$

$$
\lambda_1\lambda_2=\det(A).
$$

Here,

$$
5+2=7,
$$

and

$$
5(2)=10.
$$

These identities are useful checks, not substitutes for finding eigenvectors.


## 7. Triangular matrices

If a matrix is triangular, its eigenvalues are its diagonal entries.

For example,

$$
T=
\begin{bmatrix}
2&4&1\\
0&-1&3\\
0&0&5
\end{bmatrix}.
$$

Then

$$
T-\lambda I
=
\begin{bmatrix}
2-\lambda&4&1\\
0&-1-\lambda&3\\
0&0&5-\lambda
\end{bmatrix}.
$$

The determinant of a triangular matrix is the product of diagonal entries:

$$
\det(T-\lambda I)
=(2-\lambda)(-1-\lambda)(5-\lambda).
$$

The roots are `2`, `-1`, and `5`, exactly the diagonal entries.


## 8. Multiplicity and diagonalizability preview

The **algebraic multiplicity** of an eigenvalue is the number of times it appears as a root of the characteristic polynomial.

The **geometric multiplicity** is the dimension of its eigenspace:

$$
\dim\operatorname{Null}(A-\lambda I).
$$

Always,

$$
1\leq\text{geometric multiplicity}
\leq\text{algebraic multiplicity}.
$$

A repeated eigenvalue may provide several independent eigenvectors or only one. This difference determines whether the matrix can be diagonalized.


> **Common mistakes in Section 2**
>
> - Forming `A-lambda` instead of subtracting `lambda` from diagonal entries only.
> - Forgetting to set the determinant equal to zero.
> - Stopping after finding eigenvalues and not finding eigenvectors.
> - Solving the same system for every eigenvalue without substituting the correct value.
> - Treating a free parameter as an error; it describes the eigenvector family.
> - Confusing algebraic and geometric multiplicities.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Identity matrix | Square matrix that leaves vectors unchanged |
| Singular matrix | Square matrix with determinant zero |
| Characteristic polynomial | Determinant of `A-lambda I` |
| Characteristic equation | Characteristic polynomial set equal to zero |
| Trace | Sum of diagonal entries |
| Algebraic multiplicity | Repetition count as a polynomial root |
| Geometric multiplicity | Dimension of the corresponding eigenspace |


## Section 2 worksheet

For

$$
A=
\begin{bmatrix}
3&1\\
1&3
\end{bmatrix},
$$

1. form `A-lambda I`;
2. compute and factor the characteristic polynomial;
3. find both eigenvalues;
4. find one eigenvector for each eigenvalue;
5. verify the trace and determinant checks.


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
### 1–2. Characteristic polynomial

$$
A-\lambda I=
\begin{bmatrix}
3-\lambda&1\\1&3-\lambda
\end{bmatrix}.
$$

$$
\det(A-\lambda I)
=(3-\lambda)^2-1
=9-6\lambda+\lambda^2-1.
$$

$$
=\lambda^2-6\lambda+8
=(\lambda-4)(\lambda-2).
$$

### 3. Eigenvalues

$$
\lambda_1=4,
\qquad
\lambda_2=2.
$$

### 4. Eigenvectors

For eigenvalue `4`, the equation is `-x+y=0`, so choose

$$
\mathbf{v}_1=\begin{bmatrix}1\\1\end{bmatrix}.
$$

For eigenvalue `2`, the equation is `x+y=0`, so choose

$$
\mathbf{v}_2=\begin{bmatrix}1\\-1\end{bmatrix}.
$$

### 5. Checks

The trace is `6`, equal to `4+2`. The determinant is `8`, equal to `4(2)`.
""")


---

# Section 3: Eigendecomposition and Matrix Powers

## Learning objectives

Students should be able to construct `P` and `D`, explain diagonalizability, verify an eigendecomposition, and use it to compute matrix powers and repeated transformations.


## 1. What is eigendecomposition?

Suppose an `n by n` matrix has `n` linearly independent eigenvectors. Place them into the columns of

$$
P=
\begin{bmatrix}
\mathbf{v}_1&\cdots&\mathbf{v}_n
\end{bmatrix}.
$$

Place the corresponding eigenvalues on the diagonal of

$$
D=
\begin{bmatrix}
\lambda_1&&0\\
&\ddots&\\
0&&\lambda_n
\end{bmatrix}.
$$

Because the eigenvectors are independent, `P` is invertible. The **eigendecomposition** is

$$
A=PDP^{-1}.
$$

The matrix is then called **diagonalizable**.

Interpretation:

1. `P` inverse converts ordinary coordinates into eigenvector coordinates;
2. `D` scales each eigen-coordinate independently; and
3. `P` converts back to the original coordinate system.


## 2. Why `AP=PD`

Since every column of `P` is an eigenvector,

$$
AP
=A
\begin{bmatrix}
\mathbf{v}_1&\cdots&\mathbf{v}_n
\end{bmatrix}.
$$

Matrix multiplication acts on each column:

$$
=
\begin{bmatrix}
A\mathbf{v}_1&\cdots&A\mathbf{v}_n
\end{bmatrix}.
$$

Use the eigenvector equations:

$$
=
\begin{bmatrix}
\lambda_1\mathbf{v}_1&\cdots&\lambda_n\mathbf{v}_n
\end{bmatrix}.
$$

Multiplying `P` by the diagonal matrix `D` scales its columns in exactly this way, so

$$
AP=PD.
$$

Multiply on the right by `P` inverse:

$$
A=PDP^{-1}.
$$


## 3. Detailed eigendecomposition

Return to

$$
A=
\begin{bmatrix}
4&1\\
2&3
\end{bmatrix}.
$$

Use eigenvectors and matching eigenvalues in the same column order:

$$
P=
\begin{bmatrix}
1&1\\
1&-2
\end{bmatrix},
\qquad
D=
\begin{bmatrix}
5&0\\
0&2
\end{bmatrix}.
$$

The determinant of `P` is

$$
\det(P)=1(-2)-1(1)=-3.
$$

Therefore,

$$
P^{-1}
=\frac{1}{-3}
\begin{bmatrix}
-2&-1\\
-1&1
\end{bmatrix}
=
\begin{bmatrix}
\frac{2}{3}&\frac{1}{3}\\
\frac{1}{3}&-\frac{1}{3}
\end{bmatrix}.
$$


Compute the first product:

$$
PD
=
\begin{bmatrix}
1&1\\1&-2
\end{bmatrix}
\begin{bmatrix}
5&0\\0&2
\end{bmatrix}
=
\begin{bmatrix}
5&2\\5&-4
\end{bmatrix}.
$$

Now multiply by `P` inverse:

$$
PDP^{-1}
=
\begin{bmatrix}
5&2\\5&-4
\end{bmatrix}
\begin{bmatrix}
\frac{2}{3}&\frac{1}{3}\\
\frac{1}{3}&-\frac{1}{3}
\end{bmatrix}.
$$

Compute every entry:

$$
(1,1):\quad 5\left(\frac{2}{3}\right)+2\left(\frac{1}{3}\right)=\frac{12}{3}=4,
$$

$$
(1,2):\quad 5\left(\frac{1}{3}\right)+2\left(-\frac{1}{3}\right)=\frac{3}{3}=1,
$$

$$
(2,1):\quad 5\left(\frac{2}{3}\right)-4\left(\frac{1}{3}\right)=\frac{6}{3}=2,
$$

$$
(2,2):\quad 5\left(\frac{1}{3}\right)+(-4)\left(-\frac{1}{3}\right)=\frac{9}{3}=3.
$$

Thus,

$$
PDP^{-1}=
\begin{bmatrix}
4&1\\2&3
\end{bmatrix}=A.
$$


In [ ]:
P_eig = np.array([[1.0, 1.0], [1.0, -2.0]])
D_eig = np.diag([5.0, 2.0])
P_inv_eig = np.linalg.inv(P_eig)
A_reconstructed = P_eig @ D_eig @ P_inv_eig

print("P =\n", P_eig)
print("D =\n", D_eig)
print("P inverse =\n", P_inv_eig)
print("P D P inverse =\n", A_reconstructed)
print("Matches A:", np.allclose(A_reconstructed, A_char))


## 4. Matrix powers become diagonal powers

Starting from

$$
A=PDP^{-1},
$$

square the matrix:

$$
A^2=(PDP^{-1})(PDP^{-1}).
$$

Associativity allows the middle factors to combine:

$$
=PD(P^{-1}P)DP^{-1}.
$$

Since

$$
P^{-1}P=I,
$$

we obtain

$$
A^2=PD^2P^{-1}.
$$

Repeating the cancellation gives

$$
A^k=PD^kP^{-1}.
$$

Because `D` is diagonal,

$$
D^k=
\begin{bmatrix}
\lambda_1^k&&0\\
&\ddots&\\
0&&\lambda_n^k
\end{bmatrix}.
$$

This turns repeated matrix multiplication into repeated scalar powers.


In [ ]:
for k in [1, 2, 3, 5]:
    direct = np.linalg.matrix_power(A_char.astype(float), k)
    via_eigen = P_eig @ np.linalg.matrix_power(D_eig, k) @ P_inv_eig
    print(f"k={k}: agreement =", np.allclose(direct, via_eigen))
    print(direct)


## 5. When eigendecomposition fails

Consider

$$
J=
\begin{bmatrix}
1&1\\
0&1
\end{bmatrix}.
$$

The characteristic polynomial is

$$
(1-\lambda)^2,
$$

so the only eigenvalue is `1` with algebraic multiplicity `2`.

Solve

$$
(J-I)\mathbf{v}=\mathbf{0}.
$$

$$
J-I=
\begin{bmatrix}
0&1\\0&0
\end{bmatrix}.
$$

The equation forces

$$
y=0,
$$

while `x` is free. The eigenspace is

$$
\operatorname{span}
\left\{
\begin{bmatrix}1\\0\end{bmatrix}
\right\},
$$

which has dimension one. A two-by-two diagonalization needs two independent eigenvectors, so `J` is not diagonalizable.

Not every square matrix has an eigendecomposition, although every real matrix has a singular value decomposition, as Module 4 will show.


> **Common mistakes in Section 3**
>
> - Placing eigenvalues in a different order from their eigenvector columns.
> - Assuming `P` is automatically invertible.
> - Replacing `P` inverse with `P` transpose for a general matrix.
> - Computing `A` to the `k`th power as `P D` to the `k`th power times `P` inverse without showing the cancellations.
> - Assuming every repeated eigenvalue provides enough eigenvectors.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Diagonal matrix | Matrix whose off-diagonal entries are zero |
| Diagonalizable | Has enough independent eigenvectors for `A=PDP` inverse |
| Eigenbasis | Basis consisting entirely of eigenvectors |
| Change of coordinates | Re-expression of a vector in another basis |
| Defective matrix | Matrix lacking a complete independent eigenvector set |
| Matrix power | Repeated multiplication of a square matrix by itself |


## Section 3 worksheet

Using the eigenpairs

$$
\lambda_1=4,
\quad
\mathbf{v}_1=
\begin{bmatrix}1\\1\end{bmatrix},
\qquad
\lambda_2=2,
\quad
\mathbf{v}_2=
\begin{bmatrix}1\\-1\end{bmatrix},
$$

for

$$
A=
\begin{bmatrix}3&1\\1&3\end{bmatrix},
$$

1. form `P` and `D`;
2. compute `P` inverse;
3. verify `A=PDP` inverse;
4. write a formula for `A` to the `k`th power.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
Choose

$$
P=\begin{bmatrix}1&1\\1&-1\end{bmatrix},
\qquad
D=\begin{bmatrix}4&0\\0&2\end{bmatrix}.
$$

The determinant of `P` is `-2`, so

$$
P^{-1}=\frac{1}{2}
\begin{bmatrix}1&1\\1&-1\end{bmatrix}.
$$

Then direct multiplication gives

$$
PDP^{-1}=\begin{bmatrix}3&1\\1&3\end{bmatrix}=A.
$$

Finally,

$$
A^k
=P
\begin{bmatrix}4^k&0\\0&2^k\end{bmatrix}
P^{-1}.
$$
""")


---

# Section 4: Principal Component Analysis

## Learning objectives

Students should be able to explain centering and covariance, identify principal directions as covariance eigenvectors, compute principal-component scores, reconstruct an approximation, and distinguish dimensionality reduction from arbitrary feature deletion.


## 1. What is PCA?

**Principal component analysis**, abbreviated PCA, is a linear dimensionality-reduction method. It finds orthogonal directions that capture as much variation in centered data as possible.

PCA does not simply select original columns. It creates new coordinates called **principal components**, each formed as a linear combination of the original features.

The main steps are:

1. arrange observations in rows;
2. compute the feature mean;
3. center the data by subtracting the mean;
4. compute a covariance matrix;
5. find covariance eigenvalues and eigenvectors;
6. order eigenvectors from largest to smallest eigenvalue;
7. project the centered data onto selected eigenvectors.


## 2. Centering and covariance

Let

$$
X\in\mathbb{R}^{m\times n}
$$

contain `m` observations and `n` features. The feature-mean vector is

$$
\boldsymbol{\mu}
=\frac{1}{m}\sum_{i=1}^{m}\mathbf{x}_i.
$$

The centered data matrix is

$$
X_c=X-\mathbf{1}\boldsymbol{\mu}^T.
$$

Using the population convention in this example, the covariance matrix is

$$
C=\frac{1}{m}X_c^TX_c.
$$

The diagonal entries are feature variances. The off-diagonal entries are covariances, which measure whether features tend to move together.

Some statistics courses use `m-1` instead of `m`. That changes eigenvalues by a common scale factor but does not change eigenvector directions.


### Scaling and data-leakage caution

PCA is sensitive to measurement scale. A feature measured in thousands can dominate a feature measured in tenths even when it is not more informative. Standardization may be appropriate, but it changes the question from “largest raw variance” to “largest variance after rescaling.”

In a predictive workflow, estimate the mean, scale, and PCA directions using the training set only, then apply those fixed quantities to validation or test data. Fitting preprocessing on all data leaks information from evaluation data into training.


## 3. Detailed two-feature dataset

Consider five observations:

$$
(1,1),\quad(2,2),\quad(3,2),\quad(4,4),\quad(5,5).
$$

The first-feature mean is

$$
\overline{x}=\frac{1+2+3+4+5}{5}=3.
$$

The second-feature mean is

$$
\overline{y}=\frac{1+2+2+4+5}{5}=\frac{14}{5}=2.8.
$$

Thus,

$$
\boldsymbol{\mu}=
\begin{bmatrix}3&2.8\end{bmatrix}.
$$

The centered rows are

$$
(-2,-1.8),\quad(-1,-0.8),\quad(0,-0.8),\quad(1,1.2),\quad(2,2.2).
$$


Compute covariance entries:

$$
C_{11}
=\frac{(-2)^2+(-1)^2+0^2+1^2+2^2}{5}
=\frac{10}{5}=2.
$$

$$
C_{22}
=\frac{(-1.8)^2+(-0.8)^2+(-0.8)^2+(1.2)^2+(2.2)^2}{5}.
$$

$$
=\frac{3.24+0.64+0.64+1.44+4.84}{5}
=\frac{10.8}{5}=2.16.
$$

$$
C_{12}=C_{21}
=\frac{(-2)(-1.8)+(-1)(-0.8)+0(-0.8)+1(1.2)+2(2.2)}{5}.
$$

$$
=\frac{3.6+0.8+0+1.2+4.4}{5}
=\frac{10}{5}=2.
$$

Therefore,

$$
C=
\begin{bmatrix}
2&2\\
2&2.16
\end{bmatrix}.
$$


In [ ]:
X_pca = np.array([[1,1], [2,2], [3,2], [4,4], [5,5]], dtype=float)
mean_pca = X_pca.mean(axis=0)
X_centered_pca = X_pca - mean_pca
C_pca = (X_centered_pca.T @ X_centered_pca) / X_pca.shape[0]

eigvals_pca, eigvecs_pca = np.linalg.eigh(C_pca)
order_pca = np.argsort(eigvals_pca)[::-1]
eigvals_pca = eigvals_pca[order_pca]
eigvecs_pca = eigvecs_pca[:, order_pca]

print("Mean:", mean_pca)
print("Centered data:\n", X_centered_pca)
print("Covariance matrix:\n", C_pca)
print("Eigenvalues, largest first:", eigvals_pca)
print("Eigenvectors stored as columns:\n", eigvecs_pca)
print("Explained variance proportions:", eigvals_pca/eigvals_pca.sum())


## 4. Principal directions and explained variance

The covariance matrix is symmetric, so it has orthonormal eigenvectors and nonnegative eigenvalues.

The eigenvector associated with the largest eigenvalue is the **first principal direction**. Its eigenvalue equals the variance of the data along that direction.

For this dataset, the largest eigenvalue is approximately

$$
4.0816,
$$

and the smaller eigenvalue is approximately

$$
0.0784.
$$

The proportion explained by the first direction is approximately

$$
\frac{4.0816}{4.0816+0.0784}
\approx 0.9812.
$$

Thus one principal coordinate preserves about `98.12` percent of the total variance in this two-feature example.


## 5. Projection scores and reconstruction

Let the first unit eigenvector be

$$
\mathbf{v}_1.
$$

The one-dimensional **principal-component scores** are

$$
\mathbf{t}=X_c\mathbf{v}_1.
$$

Each score tells how far one centered observation lies along the principal direction.

The rank-one centered reconstruction is

$$
\widehat{X}_c=\mathbf{t}\mathbf{v}_1^T.
$$

Add the mean back:

$$
\widehat{X}=\widehat{X}_c+\mathbf{1}\boldsymbol{\mu}^T.
$$

The reconstruction lies on a line through the mean. It is not identical to the original data, but it retains the dominant direction.


In [ ]:
pc1 = eigvecs_pca[:, 0]
scores_pca = X_centered_pca @ pc1
X_reconstructed_pca = np.outer(scores_pca, pc1) + mean_pca

print("First principal direction:", pc1)
print("One-dimensional scores:", scores_pca)
print("Rank-one reconstruction:\n", X_reconstructed_pca)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_pca[:,0], X_pca[:,1], s=80, color="#4c78a8", label="original data")
ax.scatter(X_reconstructed_pca[:,0], X_reconstructed_pca[:,1], s=70,
           facecolors="none", edgecolors="#e45756", label="rank-one reconstruction")
line_t = np.linspace(-4, 4, 100)
line_points = mean_pca + np.outer(line_t, pc1)
ax.plot(line_points[:,0], line_points[:,1], color="#54a24b", label="first principal direction")
for original, reconstructed in zip(X_pca, X_reconstructed_pca):
    ax.plot([original[0], reconstructed[0]], [original[1], reconstructed[1]],
            color="gray", linestyle="--", linewidth=1)
ax.scatter(*mean_pca, color="black", marker="x", s=100, label="mean")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("PCA projects data onto the direction of greatest variance")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()


## 6. Why the covariance eigenvector maximizes variance

For a unit direction `v`, the projected scores are

$$
X_c\mathbf{v}.
$$

Their population variance is

$$
\frac{1}{m}\lVert X_c\mathbf{v}\rVert_2^2.
$$

Rewrite using a dot product:

$$
=\frac{1}{m}(X_c\mathbf{v})^T(X_c\mathbf{v}).
$$

Use transpose reversal:

$$
=\mathbf{v}^T\left(\frac{1}{m}X_c^TX_c\right)\mathbf{v}.
$$

Therefore,

$$
\text{projected variance}=\mathbf{v}^TC\mathbf{v}.
$$

For a symmetric covariance matrix with orthonormal eigenvectors, write `v` as a combination of those eigenvectors. The expression becomes a weighted average of the eigenvalues, with nonnegative weights summing to one. It is maximized by placing all weight on the largest eigenvalue's eigenvector. Hence the first principal direction is the dominant covariance eigenvector.


> **PCA cautions**
>
> - PCA is sensitive to feature scale. Standardization may be necessary.
> - PCA captures variance, not necessarily usefulness for a prediction task.
> - Principal components are linear combinations and may be harder to interpret.
> - Centering is essential when PCA is derived from covariance.
> - Eigenvector signs are arbitrary; reversing a principal direction reverses scores but not the represented subspace.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Centering | Subtracting each feature mean |
| Covariance | Measure of how two centered features vary together |
| Covariance matrix | Matrix containing all feature variances and covariances |
| Principal direction | Covariance eigenvector ordered by eigenvalue size |
| Principal-component score | Coordinate of an observation along a principal direction |
| Explained variance | Variance captured by a principal direction |
| Dimensionality reduction | Representing data with fewer coordinates |
| Reconstruction | Approximation mapped back to the original feature space |


## Section 4 worksheet

1. Explain why uncentered data may cause the first direction to reflect the mean location rather than variation around the mean.
2. If covariance eigenvalues are `9`, `3`, and `1`, compute the variance proportion explained by the first component and by the first two together.
3. What shape is the covariance matrix for a dataset with `200` observations and `10` features?
4. Distinguish a principal direction from a principal-component score.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. PCA is intended to measure variation around the center. Without centering, large mean offsets contribute to products and may dominate the direction calculation.

2. Total variance is

$$
9+3+1=13.
$$

The first component explains

$$
\frac{9}{13}\approx 69.23\%.
$$

The first two explain

$$
\frac{9+3}{13}=\frac{12}{13}\approx 92.31\%.
$$

3. The covariance matrix compares features with features, so its shape is `10 by 10`.

4. A principal direction is a vector in feature space. A score is one observation's scalar coordinate along that direction.
""")


---

# Section 5: Repeated Dynamics, Spectral Radius, and RNN Stability

## Learning objectives

Students should be able to solve eigenmodes of a repeated linear recurrence, define spectral radius, explain vanishing and exploding behavior, and state why eigenvalues alone may not fully describe neural-network gradients.


## 1. Recurrent update

A simple linear recurrence is

$$
\mathbf{h}_{t+1}=A\mathbf{h}_t.
$$

The vector `h_t` is the **state** at time `t`. Repeated substitution gives

$$
\mathbf{h}_t=A^t\mathbf{h}_0.
$$

A recurrent neural network, abbreviated RNN, uses a hidden state that is updated repeatedly. Real RNNs include inputs, biases, and nonlinear activations, but the linear recurrence isolates the repeated-matrix effect.


## 2. Eigenmodes of a recurrence

If the initial state is an eigenvector,

$$
\mathbf{h}_0=\mathbf{v},
$$

then

$$
\mathbf{h}_t=A^t\mathbf{v}=\lambda^t\mathbf{v}.
$$

The direction remains fixed, and only the scalar amplitude changes.

- If the eigenvalue magnitude is below one, the mode decays.
- If the eigenvalue magnitude exceeds one, the mode grows.
- If the eigenvalue magnitude equals one, it does not exponentially grow or decay.
- A negative eigenvalue alternates direction.
- A complex eigenvalue can create rotation combined with growth or decay.


## 3. Spectral radius

The **spectral radius** of a square matrix is the largest eigenvalue magnitude:

$$
\rho(A)=\max_i|\lambda_i|.
$$

For a diagonalizable matrix, long-run repeated behavior is usually dominated by eigenmodes whose eigenvalues have magnitude equal or close to the spectral radius.

For any fixed finite-dimensional matrix, the exact asymptotic statement is

$$
\rho(A)<1\quad\Longleftrightarrow\quad A^t\to 0.
$$

If $$\rho(A)>1,$$ at least one eigenmode grows exponentially. The boundary case $$\rho(A)=1$$ requires more information: a nontrivial Jordan block can produce polynomial growth even though no eigenvalue has magnitude greater than one.


In [ ]:
A_decay = np.array([[0.8, 0.0], [0.0, 0.5]])
A_explode = np.array([[1.1, 0.0], [0.0, 0.8]])
h0_dynamics = np.array([1.0, 1.0])
steps = 35

def state_norms(A, h0, steps):
    h = h0.copy()
    norms = [np.linalg.norm(h)]
    states = [h.copy()]
    for _ in range(steps):
        h = A @ h
        states.append(h.copy())
        norms.append(np.linalg.norm(h))
    return np.array(states), np.array(norms)

states_decay, norms_decay = state_norms(A_decay, h0_dynamics, steps)
states_explode, norms_explode = state_norms(A_explode, h0_dynamics, steps)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(range(steps+1), norms_decay, label="spectral radius 0.8: decay", linewidth=3)
ax.semilogy(range(steps+1), norms_explode, label="spectral radius 1.1: growth", linewidth=3)
ax.set_xlabel("time step")
ax.set_ylabel("state L2 norm on log scale")
ax.set_title("Repeated matrix multiplication can vanish or explode")
ax.legend()
plt.show()


## 4. Connection to vanishing and exploding gradients

A **Jacobian matrix** collects the partial derivatives of every output coordinate with respect to every input coordinate. It is the local linear approximation describing how a small change in one vector propagates to another vector.

During backpropagation through many recurrent steps, gradients contain products of many Jacobian matrices. In a simplified constant linear case, this resembles powers of a matrix or its transpose.

Repeated Jacobians whose relevant singular values or operator norms contract can create **vanishing gradients**, in which early-step learning signals become extremely small. Repeated Jacobians with strong expanding directions can create **exploding gradients**, in which signals become numerically huge and training becomes unstable.

Eigenvalues help describe repeated modes of a fixed matrix, but the complete gradient story is more subtle:

1. nonlinear activations change the Jacobian from step to step;
2. matrices may not be diagonalizable;
3. eigenvectors may be poorly conditioned, meaning the eigenvector basis is nearly dependent and coordinate conversion can strongly magnify numerical errors; and
4. a non-normal matrix can show large temporary growth even if every eigenvalue lies inside the unit circle.

Module 4 introduces singular values, which directly measure maximum one-step stretching and are often more informative for gradient norms.


## 5. A non-normal caution

A matrix is **normal** when it commutes with its transpose in the real case:

$$
AA^T=A^TA.
$$

Consider

$$
N=
\begin{bmatrix}
0.9&8\\
0&0.9
\end{bmatrix}.
$$

Both eigenvalues are `0.9`, so the spectral radius is below one. Long-run powers eventually decay. However, the large off-diagonal entry can create substantial temporary growth before decay becomes dominant.

This phenomenon is called **transient growth**. It shows why eigenvalues alone should not be presented as a complete training-stability test.


In [ ]:
# The spectral-radius boundary and transient growth.
decay_matrix3 = np.array([[0.9, 0.0], [0.0, 0.7]])
jordan_boundary3 = np.array([[1.0, 1.0], [0.0, 1.0]])
non_normal3 = np.array([[0.9, 8.0], [0.0, 0.9]])
powers3 = np.arange(0, 31)

def power_norms3(matrix):
    return np.array([np.linalg.norm(np.linalg.matrix_power(matrix, int(k)), 2)
                     for k in powers3])

fig, ax = plt.subplots()
ax.semilogy(powers3, power_norms3(decay_matrix3), label="spectral radius below one")
ax.semilogy(powers3, power_norms3(jordan_boundary3), label="spectral radius one: Jordan growth")
ax.semilogy(powers3, power_norms3(non_normal3), label="non-normal transient growth")
ax.set_xlabel("power t")
ax.set_ylabel("operator norm of matrix power")
ax.set_title("Eigenvalues determine asymptotics, but not every transient")
ax.legend()
plt.show()


The first curve decays. The Jordan example has spectral radius one but grows roughly linearly. The non-normal example eventually decays because its spectral radius is below one, yet it first amplifies some vectors. This separates asymptotic stability from short-term numerical behavior.


In [ ]:
N_nonnormal = np.array([[0.9, 8.0], [0.0, 0.9]])
h0_nonnormal = np.array([0.0, 1.0])
states_nonnormal, norms_nonnormal = state_norms(N_nonnormal, h0_nonnormal, 60)

print("Eigenvalues:", np.linalg.eigvals(N_nonnormal))
print("Spectral radius:", max(abs(np.linalg.eigvals(N_nonnormal))))
print("Initial norm:", norms_nonnormal[0])
print("Largest observed norm:", norms_nonnormal.max())
print("Time of largest norm:", np.argmax(norms_nonnormal))
print("Norm at final plotted time:", norms_nonnormal[-1])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(len(norms_nonnormal)), norms_nonnormal, linewidth=3, color="#e45756")
ax.axhline(norms_nonnormal[0], color="gray", linestyle="--", label="initial norm")
ax.set_xlabel("time step")
ax.set_ylabel("state L2 norm")
ax.set_title("Transient growth despite eigenvalue magnitudes below one")
ax.legend()
plt.show()


> **Common mistakes in Section 5**
>
> - Using the eigenvalue itself instead of its absolute value for growth tests.
> - Claiming spectral radius below one prevents all temporary growth.
> - Treating a nonlinear RNN as though it used one constant matrix power.
> - Confusing hidden-state behavior with gradient behavior.
> - Ignoring complex eigenvalues or defective matrices.
> - Presenting eigenvalues as the only measure of numerical stability.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Recurrent state | Vector updated from one time step to the next |
| Eigenmode | Component evolving as an eigenvalue power |
| Spectral radius | Largest eigenvalue magnitude |
| Jacobian matrix | Matrix of partial derivatives for a vector-valued transformation |
| Vanishing gradient | Learning signal shrinking toward numerical insignificance |
| Exploding gradient | Learning signal growing to unstable magnitude |
| Ill-conditioned eigenbasis | Nearly dependent eigenvector basis that can amplify errors |
| Normal matrix | Matrix commuting with its transpose in the real case |
| Non-normal matrix | Matrix that may show transient amplification not predicted by eigenvalues alone |
| Transient growth | Temporary growth before long-run decay |


## Section 5 worksheet

1. A diagonal recurrent matrix has eigenvalues `0.7` and `-1.2`. Describe both modes.
2. Compute its spectral radius.
3. Which mode dominates long-run magnitude for a state containing both directions?
4. Explain why a negative eigenvalue can still produce exploding magnitude.
5. State one reason eigenvalues alone do not completely determine RNN gradient behavior.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. The mode with eigenvalue `0.7` decays without reversing. The mode with eigenvalue `-1.2` reverses direction every step and grows in magnitude by a factor of `1.2` per step.

2. The spectral radius is

$$
\max(|0.7|,|-1.2|)=1.2.
$$

3. The `-1.2` mode dominates because its magnitude is larger.

4. Growth depends on absolute value. Although the sign alternates, the magnitude satisfies

$$
|-1.2|^t=1.2^t.
$$

5. Acceptable reasons include changing nonlinear Jacobians, non-normal transient growth, non-diagonalizability, or poor eigenvector conditioning.
""")


---

# Integrated Instructor Lab: Exact Evolution in Eigenvector Coordinates

Use

$$
A=
\begin{bmatrix}
4&1\\2&3
\end{bmatrix},
\qquad
\mathbf{h}_0=
\begin{bmatrix}
2\\-1
\end{bmatrix}.
$$

The eigenvectors are

$$
\mathbf{v}_1=
\begin{bmatrix}1\\1\end{bmatrix}
\quad\text{with eigenvalue }5,
$$

and

$$
\mathbf{v}_2=
\begin{bmatrix}1\\-2\end{bmatrix}
\quad\text{with eigenvalue }2.
$$

## Lab tasks

1. Express the initial state as a linear combination of eigenvectors.
2. Derive a closed formula for the state at time `t`.
3. Compute the states at times `1`, `2`, and `5`.
4. Explain which direction dominates as time grows.


## Detailed solution

Solve

$$
\mathbf{h}_0=c_1\mathbf{v}_1+c_2\mathbf{v}_2.
$$

That is,

$$
\begin{bmatrix}2\\-1\end{bmatrix}
=c_1\begin{bmatrix}1\\1\end{bmatrix}
+c_2\begin{bmatrix}1\\-2\end{bmatrix}.
$$

The coordinate equations are

$$
c_1+c_2=2,
$$

$$
c_1-2c_2=-1.
$$

Subtract the second equation from the first:

$$
3c_2=3,
$$

so

$$
c_2=1.
$$

Then

$$
c_1=1.
$$

Therefore,

$$
\mathbf{h}_0=\mathbf{v}_1+\mathbf{v}_2.
$$


Apply `A` repeatedly to each eigenmode:

$$
\mathbf{h}_t=A^t\mathbf{h}_0
=A^t(\mathbf{v}_1+\mathbf{v}_2).
$$

Use linearity:

$$
=A^t\mathbf{v}_1+A^t\mathbf{v}_2.
$$

Use the eigenmode formula:

$$
\mathbf{h}_t=5^t\mathbf{v}_1+2^t\mathbf{v}_2.
$$

Thus,

$$
\mathbf{h}_t
=5^t
\begin{bmatrix}1\\1\end{bmatrix}
+2^t
\begin{bmatrix}1\\-2\end{bmatrix}
=
\begin{bmatrix}
5^t+2^t\\
5^t-2^{t+1}
\end{bmatrix}.
$$

Because

$$
5^t
$$

grows faster than

$$
2^t,
$$

the state direction approaches the eigendirection of

$$
\begin{bmatrix}1\\1\end{bmatrix}.
$$


In [ ]:
h0_lab3 = np.array([2.0, -1.0])
v1_lab3 = np.array([1.0, 1.0])
v2_lab3 = np.array([1.0, -2.0])

for t in [0, 1, 2, 5]:
    direct = np.linalg.matrix_power(A_char, t) @ h0_lab3
    eigen_formula = (5**t)*v1_lab3 + (2**t)*v2_lab3
    print(f"t={t}: direct={direct}, eigen formula={eigen_formula}, agree={np.allclose(direct, eigen_formula)}")

states_lab3 = np.array([np.linalg.matrix_power(A_char, t) @ h0_lab3 for t in range(8)])
directions_lab3 = states_lab3 / np.linalg.norm(states_lab3, axis=1, keepdims=True)
print("Normalized directions over time:\n", directions_lab3)


---

# Module 3 Wrap-Up

## Mathematical storyline

1. Eigenvectors are nonzero directions that a square matrix scales without turning.
2. Eigenvalues quantify scaling, reversal, collapse, growth, or decay along those directions.
3. The characteristic equation finds values that make a nonzero homogeneous solution possible.
4. Eigendecomposition changes coordinates to an eigenbasis, applies independent scalings, and changes back.
5. Diagonal powers make repeated dynamics transparent.
6. PCA uses covariance eigenvectors to find directions of greatest centered variation.
7. Recurrent eigenmodes explain part of the vanishing and exploding behavior story.
8. Spectral radius is useful but does not capture all transient or nonlinear effects.


## Formula reference

### Eigenvector equation

$$
A\mathbf{v}=\lambda\mathbf{v},
\qquad
\mathbf{v}\neq\mathbf{0}.
$$

### Characteristic equation

$$
\det(A-\lambda I)=0.
$$

### Eigendecomposition

$$
A=PDP^{-1}.
$$

### Matrix powers

$$
A^k=PD^kP^{-1}.
$$

### Population covariance

$$
C=\frac{1}{m}X_c^TX_c.
$$

### Spectral radius

$$
\rho(A)=\max_i|\lambda_i|.
$$


## Cumulative practice

Let

$$
A=
\begin{bmatrix}
2&0\\
1&3
\end{bmatrix}.
$$

1. Use the triangular structure to identify the eigenvalues.
2. Find one eigenvector for each eigenvalue.
3. Construct `P` and `D`.
4. Determine whether the matrix is diagonalizable.
5. State the spectral radius.
6. Predict the dominant long-run eigendirection for a generic initial vector containing both modes.


In [ ]:
reveal_markdown("Open cumulative solution", r"""
### 1. Eigenvalues

The matrix is triangular, so its eigenvalues are the diagonal entries:

$$
\lambda_1=2,
\qquad
\lambda_2=3.
$$

### 2. Eigenvectors

For eigenvalue `2`,

$$
A-2I=\begin{bmatrix}0&0\\1&1\end{bmatrix},
$$

so `x+y=0`. Choose

$$
\mathbf{v}_1=\begin{bmatrix}1\\-1\end{bmatrix}.
$$

For eigenvalue `3`,

$$
A-3I=\begin{bmatrix}-1&0\\1&0\end{bmatrix},
$$

so `x=0`. Choose

$$
\mathbf{v}_2=\begin{bmatrix}0\\1\end{bmatrix}.
$$

### 3–4. Decomposition

$$
P=\begin{bmatrix}1&0\\-1&1\end{bmatrix},
\qquad
D=\begin{bmatrix}2&0\\0&3\end{bmatrix}.
$$

The eigenvectors are independent, so the matrix is diagonalizable.

### 5–6. Long-run behavior

The spectral radius is `3`. A generic state containing both modes becomes dominated by the eigenvector

$$
\begin{bmatrix}0\\1\end{bmatrix}
$$

because `3` to the `t`th power grows faster than `2` to the `t`th power.
""")


## Exit ticket

1. Why is the zero vector excluded from the eigenvector definition?
2. What does the characteristic equation accomplish?
3. When is a matrix diagonalizable?
4. Why does PCA use the largest covariance eigenvalue?
5. What does spectral radius reveal, and what can it miss?
